In [1]:
import time
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
plt.style.use('ggplot')

from sklearn.ensemble import HistGradientBoostingRegressor

from cluster_experiments.random_splitter import ClusteredSplitter
from cluster_experiments.perturbator import ConstantPerturbator
from cluster_experiments.experiment_analysis import ClusteredOLSAnalysis
from cluster_experiments.power_analysis import PowerAnalysis
from cluster_experiments.power_analysis import NormalPowerAnalysis
from cluster_experiments import (
    AnalysisPlan,
    SimpleMetric,
    Dimension,
    Variant,
    HypothesisTest,
    TargetAggregation,
)

# Setup

In [2]:
np.random.seed(42)

N_CUSTOMERS = 50_000
MIN_DATE = pd.Timestamp('2021-01-01')
MAX_DATE = MIN_DATE + pd.Timedelta(days=360)
customer_ids = np.random.choice(np.arange(1e6, 1e7).astype(int), size=N_CUSTOMERS, replace=False)
beta_samples = np.random.beta(2, 5, size=N_CUSTOMERS)
customer_mean_time_between_orders = 7 + (60 - 7) * (1 - beta_samples)  # Invert to favor larger values
customer_average_spend = np.random.uniform(20, 200, size=N_CUSTOMERS)
start_times = np.random.choice(pd.date_range(MIN_DATE.replace(year=2020), MAX_DATE.replace(year=2020)), size=N_CUSTOMERS, replace=True)
orders = []
for customer_id, start_time, time_between_orders, avg_spend in zip(
    customer_ids,
    start_times,
    customer_mean_time_between_orders,
    customer_average_spend
):
    order_time = start_time
    while order_time < MAX_DATE:
        order_time = order_time + pd.Timedelta(days=np.random.exponential(scale=time_between_orders))
        if order_time >= MIN_DATE:
            order_value = np.round(np.random.normal(loc=avg_spend, scale=avg_spend * 0.3), 2)
            orders.append({
                'customer_id': customer_id,
                'order_time': order_time,
                'order_value': max(0, order_value)
            })

In [3]:
data = (
    pd.DataFrame(orders)
    .sample(frac=1, random_state=42, replace=False)
    .reset_index(drop=True)
    .assign(
        time_index = lambda df: (df['order_time'] - df['order_time'].min()).dt.days
    )
)

In [4]:
data.describe(include='all').T

,count,mean,min,25%,50%,75%,max,std
customer_id,471499.0,5507872.216215,1000002.0,3256858.0,5487974.0,7756395.0,9999863.0,2597719.05622
order_time,471499,2021-07-23 17:58:30.200383744,2021-01-01 00:00:48.778392785,2021-04-11 13:18:34.628055552,2021-07-21 07:51:42.174399488,2021-10-29 23:15:26.620336128,2023-05-06 01:21:40.559308211,NaN
order_value,471499.0,109.850592,0.0,58.3,101.01,152.02,447.19,63.254347
time_index,471499.0,203.24817,0.0,100.0,201.0,301.0,855.0,121.186902
